In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("run_id", "manual")
run_id = dbutils.widgets.get("run_id")

In [0]:
df_pii = (
    spark.read
    .table("databricksbootcamp.greeengridcasestudy.bronze_pii")
    .drop("ingested_at", "run_id")
)

df_plans = (
    spark.read
    .table("databricksbootcamp.greeengridcasestudy.bronze_plans")
    .drop("ingested_at", "run_id")
)

df_readings = (
    spark.read
    .table("databricksbootcamp.greeengridcasestudy.bronze_readings")
    .filter(
        (F.col("kwh") < 0) |
        (F.col("kwh") > 100)
    )
    .drop("ingested_at", "run_id")
)

In [0]:
df_join = (
    df_readings.join(df_pii, df_readings.meter_id == df_pii.MeterID, "left")
    .drop("MeterID")
    .join(df_plans, df_pii.PlanID == df_plans.PlanID, "left")
    .drop(df_pii.PlanID)
    .withColumn("CustomerHash", F.sha2(F.col("CustomerEmail"), 256)).drop("CustomerEmail")
    .withColumn("Rejected_Reason", F.when(F.col("kwh") < 0, "negative_reading")
                .when(F.col("kwh") > 100, "over_ceiling").otherwise("unknown"))
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("run_id", F.lit(run_id))
)

In [0]:
(
df_join.write.mode("overwrite")
    .saveAsTable("databricksbootcamp.greeengridcasestudy.quarantine_radings")
)